# Sprint 5 — Collection Mapping

Spec: [`sprint-5-tasks.md`](../../litemapper/docs/requirements/sprint-5-tasks.md) — 12 tasks (S5-T00..T11) covering every common .NET collection shape.

| Shape                                            | Task   |
| ------------------------------------------------ | ------ |
| Array `T[]`                                      | S5-T01 |
| `List<T>` pre-sized                              | S5-T02 |
| `IEnumerable<T>`                                 | S5-T03 |
| `ICollection<T>` / `IReadOnlyList<T>` / …        | S5-T04 |
| `HashSet<T>`                                     | S5-T05 |
| `Dictionary<K, V>`                               | S5-T06 |
| Immutable / observable / read-only               | S5-T08 |
| Nested collections (collections of collections)  | S5-T09 |
| `Dictionary<string, object>` ↔ object            | S5-T10 |


## Setup


In [ ]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using System.Collections.Immutable;
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
Console.WriteLine("Ready.");


## 1. `MapAll<S, D>()` over `IEnumerable<S>` → `List<D>`

The simplest collection entry point — one `Bind<Line, LineDto>` covers any `IEnumerable<Line>` source.


In [ ]:
public sealed class Line    { public string Sku { get; init; } = ""; public int Qty { get; init; } }
public sealed class LineDto { public string Sku { get; set; } = ""; public int Qty { get; set; } }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Line, LineDto>(_ => { }))
    .Forge();

IEnumerable<Line> Sample()
{
    yield return new Line { Sku = "A", Qty = 1 };
    yield return new Line { Sku = "B", Qty = 2 };
    yield return new Line { Sku = "C", Qty = 3 };
}

var mapped = sculptor.MapAll<Line, LineDto>(Sample());
Console.WriteLine($"Type  : {mapped.GetType().Name}");
Console.WriteLine($"Count : {mapped.Count}");
foreach (var d in mapped) Console.WriteLine($"  {d.Sku} x{d.Qty}");


## 2. Collection-valued target members — arrays, `List<T>`, `HashSet<T>`

When a target property is a collection type, the mapper infers the construction strategy from the declared type. Arrays are stack-sized; `List<T>` is pre-sized from the origin's `Count`; `HashSet<T>` deduplicates via its equality comparer.


In [ ]:
public sealed class OrderSrc
{
    public Line[]            LinesArray { get; init; } = Array.Empty<Line>();
    public List<Line>        LinesList  { get; init; } = new();
    public HashSet<string>   Tags       { get; init; } = new();
}

public sealed class OrderDst
{
    public LineDto[]         LinesArray { get; set; } = Array.Empty<LineDto>();
    public List<LineDto>     LinesList  { get; set; } = new();
    public HashSet<string>   Tags       { get; set; } = new();
}

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Line, LineDto>(_ => { });
        o.Bind<OrderSrc, OrderDst>(_ => { });
    })
    .Forge();

var src = new OrderSrc
{
    LinesArray = new[] { new Line { Sku = "A", Qty = 1 } },
    LinesList  = new   { new Line { Sku = "B", Qty = 2 }, new Line { Sku = "C", Qty = 3 } }.ToList(),
    Tags       = new HashSet<string> { "priority", "gift", "priority" },
};

var dto = sculptor.Map<OrderSrc, OrderDst>(src);
Console.WriteLine($"LinesArray  : {dto.LinesArray.GetType().Name}, count={dto.LinesArray.Length}");
Console.WriteLine($"LinesList   : {dto.LinesList.GetType().Name}, count={dto.LinesList.Count}");
Console.WriteLine($"Tags        : {dto.Tags.GetType().Name}, count={dto.Tags.Count}  (deduplicated)");


## 3. `Dictionary<K, V>` — keys and values mapped independently

The key's transformer chain and the value's mapping pipeline run independently per entry. Dictionary order is preserved on ordered dictionaries (e.g. `SortedDictionary<K,V>`).


In [ ]:
public sealed class PriceBookSrc { public Dictionary<string, Line> BySku { get; init; } = new(); }
public sealed class PriceBookDst { public Dictionary<string, LineDto> BySku { get; set; } = new(); }

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Line, LineDto>(_ => { });
        o.Bind<PriceBookSrc, PriceBookDst>(_ => { });
    })
    .Forge();

var src = new PriceBookSrc
{
    BySku =
    {
        ["KEY-001"] = new Line { Sku = "KEY-001", Qty = 5 },
        ["MSE-001"] = new Line { Sku = "MSE-001", Qty = 2 },
    }
};

var dto = sculptor.Map<PriceBookSrc, PriceBookDst>(src);
foreach (var (sku, line) in dto.BySku)
    Console.WriteLine($"  [{sku}] = {line.Sku} x{line.Qty}");


## 4. `ImmutableList<T>` + `ImmutableArray<T>` + `ImmutableHashSet<T>`

Immutable targets go through the corresponding `.ToImmutable*()` factory. No mutability in the mapped instance.


In [ ]:
public sealed class BatchSrc { public List<Line> Lines { get; init; } = new(); }
public sealed class BatchImmutableDst
{
    public ImmutableList<LineDto>    AsList  { get; set; } = ImmutableList<LineDto>.Empty;
    public ImmutableArray<LineDto>   AsArray { get; set; }
    public ImmutableHashSet<string>  Tags    { get; set; } = ImmutableHashSet<string>.Empty;
}

// Illustrate by materialising each immutable flavour from the same origin collection.
var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Line, LineDto>(_ => { });
        o.Bind<BatchSrc, BatchImmutableDst>(rule => rule
            .Property(d => d.AsList,  p => p.From(b => b.Lines))
            .Property(d => d.AsArray, p => p.From(b => b.Lines))
            .Property(d => d.Tags,    p => p.From(b => b.Lines.Select(l => l.Sku))));
    })
    .Forge();

var src = new BatchSrc { Lines = { new Line { Sku = "A", Qty = 1 }, new Line { Sku = "B", Qty = 2 } } };
var dto = sculptor.Map<BatchSrc, BatchImmutableDst>(src);
Console.WriteLine($"AsList  : {dto.AsList.GetType().Name}, count={dto.AsList.Count}");
Console.WriteLine($"AsArray : ImmutableArray, length={dto.AsArray.Length}");
Console.WriteLine($"Tags    : {dto.Tags.GetType().Name}, count={dto.Tags.Count}");


## 5. Nested collections — `List<List<T>>`

Collections of collections recurse through the mapper. Each inner element is processed by the same per-element pipeline.


In [ ]:
public sealed class Matrix    { public List<List<int>> Cells { get; init; } = new(); }
public sealed class MatrixDto { public List<List<int>> Cells { get; set; } = new(); }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Matrix, MatrixDto>(_ => { }))
    .Forge();

var src = new Matrix { Cells = { new() { 1, 2, 3 }, new() { 4, 5, 6 }, new() { 7, 8, 9 } } };
var dto = sculptor.Map<Matrix, MatrixDto>(src);
foreach (var row in dto.Cells)
    Console.WriteLine($"  [ {string.Join(", ", row)} ]");


## Next

- **`sprint-06-polymorphism-and-blueprints.ipynb`** — inheritance hierarchies, interface targets, and reusable `MappingBlueprint` classes.
